# Assignment 3: Fine-tuning language models




In [1]:
%pip install -q "transformers==4.57.3" "datasets==4.4.1" "evaluate==0.4.6" "rouge_score==0.1.2" "accelerate==1.12.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Configuration

In [2]:
import gc
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = DEVICE == "cuda"

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
MAX_TRAIN_SAMPLES = 5_000
MAX_TEST_SAMPLES = 400
MAX_LENGTH = 512
NUM_EPOCHS = 5
EFFECTIVE_BATCH_SIZE = 16

OUTPUT_ROOT = Path("/content/a3_runs") if Path("/content").exists() else Path("./a3_runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "epochs_per_fine-tuning_run": NUM_EPOCHS,
    "train_examples": MAX_TRAIN_SAMPLES,
    "test_examples": MAX_TEST_SAMPLES,
})

{'device': 'cuda', 'gpu': 'Tesla T4', 'epochs_per_fine-tuning_run': 5, 'train_examples': 5000, 'test_examples': 400}


## Part 1: Preprocessing

### Task 1.1: Load and inspect SmolTalk

In [3]:
from datasets import Dataset, DatasetDict, load_dataset

smoltalk = load_dataset("HuggingFaceTB/smoltalk", "all", streaming=True)

def is_short_single_turn(example):
    messages = example["messages"]
    roles = [message["role"] for message in messages]
    valid_roles = roles in (["user", "assistant"], ["system", "user", "assistant"])
    short_enough = all(len(message["content"]) <= 256 for message in messages)
    return valid_roles and short_enough

def materialize_split(split_name, sample_count):
    stream = smoltalk[split_name].shuffle(seed=SEED, buffer_size=10_000)
    rows = list(stream.filter(is_short_single_turn).take(sample_count))
    if len(rows) != sample_count:
        raise RuntimeError(f"Expected {sample_count} rows from {split_name}, found {len(rows)}")
    return Dataset.from_list(rows)

smoltalk_simplified = DatasetDict({
    "train": materialize_split("train", MAX_TRAIN_SAMPLES),
    "test": materialize_split("test", MAX_TEST_SAMPLES),
})

smoltalk_simplified

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

In [4]:
for split in ("train", "test"):
    example = smoltalk_simplified[split][0]
    print(f"{split}: {len(smoltalk_simplified[split])} examples")
    print(example["messages"])
    print()

train: 5000 examples
[{'content': 'Classify the given restaurant based on cuisine type.\nThe restaurant serves Thai food, Vietnamese food, Chinese food, and Indonesian food.', 'role': 'user'}, {'content': 'The restaurant can be classified as serving Asian cuisine, specifically Southeast Asian and East Asian cuisine, as it offers dishes from Thailand, Vietnam, China, and Indonesia.', 'role': 'assistant'}]

test: 400 examples
[{'content': "Rewrite the following text in a more formal tone.\nHey, I just wanted to tell you that I got the ticket to that super amazing concert we talked about last time. It's gonna be so awesome, can't wait to go!", 'role': 'user'}, {'content': 'I would like to inform you that I have successfully acquired the ticket for the exceptional concert we discussed previously. I am eagerly anticipating the event.', 'role': 'assistant'}]



### Task 1.2: Format instruction-response pairs

In [5]:
DEFAULT_SYSTEM_PROMPT = "You are a helpful AI assistant named SmolLM, trained by Hugging Face"

def format_input_output(example):
    messages = example["messages"]

    if messages[0]["role"] == "system":
        system_text = messages[0]["content"].strip()
        user_text = messages[1]["content"].strip()
        assistant_text = messages[2]["content"].strip()
    else:
        system_text = DEFAULT_SYSTEM_PROMPT
        user_text = messages[0]["content"].strip()
        assistant_text = messages[1]["content"].strip()

    prompt = (
        f"<|im_start|>system\n{system_text}<|im_end|>\n"
        f"<|im_start|>user\n{user_text}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    response = f"{assistant_text}<|im_end|>\n"
    return {"prompt": prompt, "response": response}

ds_sft = smoltalk_simplified.map(format_input_output)
print(ds_sft["train"][0]["prompt"])
print(ds_sft["train"][0]["response"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Classify the given restaurant based on cuisine type.
The restaurant serves Thai food, Vietnamese food, Chinese food, and Indonesian food.<|im_end|>
<|im_start|>assistant

The restaurant can be classified as serving Asian cuisine, specifically Southeast Asian and East Asian cuisine, as it offers dishes from Thailand, Vietnam, China, and Indonesia.<|im_end|>



### Task 1.3: Tokenize and mask prompt labels

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_helper(example):
    prompt_ids = tokenizer(example["prompt"], add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(example["response"], add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + response_ids
    labels = [-100] * len(prompt_ids) + response_ids.copy()

    # Keep the response and the most recent prompt context if a sequence is too long.
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[-MAX_LENGTH:]
        labels = labels[-MAX_LENGTH:]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }

tokenized_ds_sft = ds_sft.map(
    tokenize_helper,
    remove_columns=ds_sft["train"].column_names,
    desc="Tokenizing",
)

sample = tokenized_ds_sft["train"][0]
assert len(sample["input_ids"]) == len(sample["attention_mask"]) == len(sample["labels"])
assert any(label == -100 for label in sample["labels"])
assert any(label != -100 for label in sample["labels"])

lengths = np.array(tokenized_ds_sft["train"]["input_ids"], dtype=object)
length_values = np.array([len(ids) for ids in lengths])
print({
    "min_length": int(length_values.min()),
    "median_length": int(np.median(length_values)),
    "max_length": int(length_values.max()),
})

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

Tokenizing:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/400 [00:00<?, ? examples/s]

{'min_length': 44, 'median_length': 98, 'max_length': 500}


In [7]:
visible_labels = [token_id for token_id in sample["labels"] if token_id != -100]
print("Model input:\n", tokenizer.decode(sample["input_ids"], skip_special_tokens=False))
print("\nLoss-bearing response:\n", tokenizer.decode(visible_labels, skip_special_tokens=False))

Model input:
 <|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Classify the given restaurant based on cuisine type.
The restaurant serves Thai food, Vietnamese food, Chinese food, and Indonesian food.<|im_end|>
<|im_start|>assistant
The restaurant can be classified as serving Asian cuisine, specifically Southeast Asian and East Asian cuisine, as it offers dishes from Thailand, Vietnam, China, and Indonesia.<|im_end|>


Loss-bearing response:
 The restaurant can be classified as serving Asian cuisine, specifically Southeast Asian and East Asian cuisine, as it offers dishes from Thailand, Vietnam, China, and Indonesia.<|im_end|>



## Part 2: Evaluate the pretrained baseline

In [8]:
def data_collator(batch):
    input_ids_list = [torch.tensor(row["input_ids"], dtype=torch.long) for row in batch]
    attention_masks_list = [torch.tensor(row["attention_mask"], dtype=torch.long) for row in batch]
    labels_list = [torch.tensor(row["labels"], dtype=torch.long) for row in batch]
    max_len = max(tensor.size(0) for tensor in input_ids_list)

    def pad_to_max(tensors, pad_value):
        padded = []
        for tensor in tensors:
            pad_len = max_len - tensor.size(0)
            if pad_len:
                tensor = torch.cat([
                    tensor,
                    torch.full((pad_len,), pad_value, dtype=tensor.dtype),
                ])
            padded.append(tensor)
        return torch.stack(padded)

    return {
        "input_ids": pad_to_max(input_ids_list, tokenizer.pad_token_id),
        "attention_mask": pad_to_max(attention_masks_list, 0),
        "labels": pad_to_max(labels_list, -100),
    }

batch = data_collator([tokenized_ds_sft["train"][0], tokenized_ds_sft["train"][1]])
print({key: tuple(value.shape) for key, value in batch.items()})

{'input_ids': (2, 92), 'attention_mask': (2, 92), 'labels': (2, 92)}


In [9]:
import evaluate

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

class RougeMetricComputer:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.predictions = []
        self.references = []

    def __call__(self, eval_pred, compute_result=False):
        prediction_ids, labels = eval_pred

        if isinstance(prediction_ids, torch.Tensor):
            prediction_ids = prediction_ids.detach().cpu()
        if isinstance(labels, torch.Tensor):
            labels = labels.detach().cpu()

        for predicted, gold in zip(prediction_ids, labels):
            # Logit t predicts label t+1 in a causal LM.
            answer_mask = gold[1:] != -100
            if not bool(answer_mask.any()):
                continue
            reference_ids = gold[1:][answer_mask]
            aligned_prediction_ids = predicted[:-1][answer_mask]
            self.references.append(
                self.tokenizer.decode(reference_ids.tolist(), skip_special_tokens=True).strip()
            )
            self.predictions.append(
                self.tokenizer.decode(aligned_prediction_ids.tolist(), skip_special_tokens=True).strip()
            )

        if not compute_result:
            return {}

        scores = self.rouge.compute(
            predictions=self.predictions,
            references=self.references,
        ) if self.references else {"rougeL": 0.0}
        self.predictions.clear()
        self.references.clear()
        return {"rougeL": float(scores["rougeL"])}

In [10]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments

def make_trainer(model, training_args):
    return Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=RougeMetricComputer(tokenizer),
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
        data_collator=data_collator,
    )

def common_training_arguments(output_dir):
    return dict(
        output_dir=str(output_dir),
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=EFFECTIVE_BATCH_SIZE // 4,
        fp16=USE_FP16,
        bf16=False,
        report_to="none",
        batch_eval_metrics=True,
        eval_accumulation_steps=8,
        save_strategy="no",
        logging_steps=100,
        seed=SEED,
        data_seed=SEED,
        optim="adamw_torch",
    )

def prepare_model(model):
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    return model

def release_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### Task 2.2: Baseline metrics

In [11]:
pretrained_model = prepare_model(
    AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
)
pretrained_total_parameters = sum(parameter.numel() for parameter in pretrained_model.parameters())

pretrained_eval_args = TrainingArguments(
    eval_strategy="no",
    **common_training_arguments(OUTPUT_ROOT / "pretrained_eval"),
)
pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

print(json.dumps(pretrained_eval_metrics, indent=2))

del pretrained_trainer, pretrained_model
release_cuda()

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

{
  "eval_loss": 2.049764633178711,
  "eval_model_preparation_time": 0.0075,
  "eval_rougeL": 0.5670730900945649,
  "eval_runtime": 11.1366,
  "eval_samples_per_second": 35.918,
  "eval_steps_per_second": 8.979
}


ROUGE-L can be non-zero before SFT because evaluation is teacher-forced, many local continuations are predictable, and the base model already learned substantial language regularities during pretraining.

## Part 3: Full supervised fine-tuning

### Task 3.1: Train all parameters

In [12]:
full_model = prepare_model(
    AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
)
full_model.config.use_cache = False

full_training_args = TrainingArguments(
    eval_strategy="epoch",
    num_train_epochs=NUM_EPOCHS,
    learning_rate=5e-5,
    warmup_ratio=0.05,
    weight_decay=0.01,
    **common_training_arguments(OUTPUT_ROOT / "full_sft"),
)
full_trainer = make_trainer(full_model, full_training_args)
full_train_metrics = full_trainer.train().metrics
full_eval_metrics = full_trainer.evaluate()
full_model.config.use_cache = True

print("Training metrics:")
print(json.dumps(full_train_metrics, indent=2))
print("Final evaluation metrics:")
print(json.dumps(full_eval_metrics, indent=2))

Epoch,Training Loss,Validation Loss,Rougel
1,1.092100,1.075575,0.680235
2,0.801700,1.061201,0.685762
3,0.587100,1.106894,0.683970
4,0.427200,1.183206,0.678547
5,0.333700,1.250820,0.675465


Training metrics:
{
  "train_runtime": 1001.0297,
  "train_samples_per_second": 24.974,
  "train_steps_per_second": 1.563,
  "total_flos": 2293594474977792.0,
  "train_loss": 0.6746453879359431,
  "epoch": 5.0
}
Final evaluation metrics:
{
  "eval_loss": 1.2508196830749512,
  "eval_rougeL": 0.6754646595110697,
  "eval_runtime": 7.4664,
  "eval_samples_per_second": 53.573,
  "eval_steps_per_second": 13.393,
  "epoch": 5.0
}


### Task 3.3: Count trainable parameters

In [13]:
def num_trainable_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

full_trainable_parameters = num_trainable_parameters(full_model)
assert full_trainable_parameters == pretrained_total_parameters
print(f"Full SFT trainable parameters: {full_trainable_parameters:,}")

del full_trainer
full_model.to("cpu")
release_cuda()

Full SFT trainable parameters: 134,515,008


## Part 4: Parameter-efficient fine-tuning

### Task 4.1: Find and replace attention projections

In [14]:
import torch.nn as nn

LORA_SUFFIXES = (
    "self_attn.q_proj",
    "self_attn.k_proj",
    "self_attn.v_proj",
    "self_attn.o_proj",
)

def extract_lora_targets(model):
    targets = {
        name: module
        for name, module in model.named_modules()
        if isinstance(module, nn.Linear) and name.endswith(LORA_SUFFIXES)
    }
    expected = model.config.num_hidden_layers * len(LORA_SUFFIXES)
    if len(targets) != expected:
        raise ValueError(f"Expected {expected} attention projections, found {len(targets)}")
    return targets

def replace_layers(model, named_layers):
    for name, layer in named_layers.items():
        components = name.split(".")
        parent = model
        for component in components[:-1]:
            parent = getattr(parent, component)
        setattr(parent, components[-1], layer)
    return model

### Task 4.2: Implement the LoRA layer

In [15]:
class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        super().__init__()
        if not isinstance(W, nn.Linear):
            raise TypeError("W must be torch.nn.Linear")
        if r <= 0:
            raise ValueError("r must be positive")

        self.W = W
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        for parameter in self.W.parameters():
            parameter.requires_grad = False

        factory_kwargs = {"device": W.weight.device, "dtype": W.weight.dtype}
        self.A = nn.Parameter(torch.empty(r, W.in_features, **factory_kwargs))
        self.B = nn.Parameter(torch.zeros(W.out_features, r, **factory_kwargs))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        base_output = self.W(x)
        update = (x @ self.A.transpose(0, 1)) @ self.B.transpose(0, 1)
        return base_output + self.scaling * update

### Task 4.3: Train with LoRA

In [16]:
LORA_RANK = 8
LORA_ALPHA = 16

lora_model = prepare_model(
    AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
)
lora_model.config.use_cache = False

for parameter in lora_model.parameters():
    parameter.requires_grad = False

lora_targets = extract_lora_targets(lora_model)
expected_lora_parameters = sum(
    LORA_RANK * (layer.in_features + layer.out_features)
    for layer in lora_targets.values()
)
lora_wrappers = {
    name: LoRALayer(layer, r=LORA_RANK, alpha=LORA_ALPHA)
    for name, layer in lora_targets.items()
}
lora_model = replace_layers(lora_model, lora_wrappers)

lora_trainable_parameters = num_trainable_parameters(lora_model)
assert lora_trainable_parameters == expected_lora_parameters
print(f"LoRA target layers: {len(lora_targets)}")
print(f"LoRA trainable parameters: {lora_trainable_parameters:,}")
print(f"Share of base parameters: {100 * lora_trainable_parameters / pretrained_total_parameters:.3f}%")

LoRA target layers: 120
LoRA trainable parameters: 921,600
Share of base parameters: 0.685%


In [17]:
lora_training_args = TrainingArguments(
    eval_strategy="epoch",
    num_train_epochs=NUM_EPOCHS,
    learning_rate=3e-4,
    warmup_ratio=0.05,
    weight_decay=0.0,
    **common_training_arguments(OUTPUT_ROOT / "lora_sft"),
)
lora_trainer = make_trainer(lora_model, lora_training_args)
lora_train_metrics = lora_trainer.train().metrics
lora_eval_metrics = lora_trainer.evaluate()
lora_model.config.use_cache = True

print("Training metrics:")
print(json.dumps(lora_train_metrics, indent=2))
print("Final evaluation metrics:")
print(json.dumps(lora_eval_metrics, indent=2))

del lora_trainer
lora_model.to("cpu")
release_cuda()

Epoch,Training Loss,Validation Loss,Rougel
1,1.289200,1.293720,0.653501
2,1.188200,1.239802,0.661372
3,1.149300,1.213579,0.664730
4,1.116000,1.205639,0.666672
5,1.108300,1.200830,0.668052


Training metrics:
{
  "train_runtime": 1176.9252,
  "train_samples_per_second": 21.242,
  "train_steps_per_second": 1.33,
  "total_flos": 2313497562388992.0,
  "train_loss": 1.202904503185528,
  "epoch": 5.0
}
Final evaluation metrics:
{
  "eval_loss": 1.2008295059204102,
  "eval_rougeL": 0.668051948893676,
  "eval_runtime": 8.5362,
  "eval_samples_per_second": 46.859,
  "eval_steps_per_second": 11.715,
  "epoch": 5.0
}


In [18]:
def safe_perplexity(loss):
    return math.exp(min(float(loss), 20.0))

comparison = pd.DataFrame([
    {
        "model": "Pretrained",
        "updated_parameters": 0,
        "train_seconds": 0.0,
        "eval_loss": pretrained_eval_metrics["eval_loss"],
        "perplexity": safe_perplexity(pretrained_eval_metrics["eval_loss"]),
        "rougeL": pretrained_eval_metrics.get("eval_rougeL", float("nan")),
    },
    {
        "model": "Full SFT",
        "updated_parameters": full_trainable_parameters,
        "train_seconds": full_train_metrics["train_runtime"],
        "eval_loss": full_eval_metrics["eval_loss"],
        "perplexity": safe_perplexity(full_eval_metrics["eval_loss"]),
        "rougeL": full_eval_metrics.get("eval_rougeL", float("nan")),
    },
    {
        "model": "LoRA SFT",
        "updated_parameters": lora_trainable_parameters,
        "train_seconds": lora_train_metrics["train_runtime"],
        "eval_loss": lora_eval_metrics["eval_loss"],
        "perplexity": safe_perplexity(lora_eval_metrics["eval_loss"]),
        "rougeL": lora_eval_metrics.get("eval_rougeL", float("nan")),
    },
])

comparison["parameter_share_percent"] = (
    100 * comparison["updated_parameters"] / pretrained_total_parameters
)
comparison.round(4)

,model,updated_parameters,train_seconds,eval_loss,perplexity,rougeL,parameter_share_percent
0,Pretrained,0,0.0000,2.0498,7.7661,0.5671,0.0000
1,Full SFT,134515008,1001.0297,1.2508,3.4932,0.6755,100.0000
2,LoRA SFT,921600,1176.9252,1.2008,3.3229,0.6681,0.6851


### Task 4.4: Qualitative inspection

In [19]:
def build_inference_prompt(user_text, system_text=None):
    system_text = system_text or DEFAULT_SYSTEM_PROMPT
    return (
        f"<|im_start|>system\n{system_text.strip()}<|im_end|>\n"
        f"<|im_start|>user\n{user_text.strip()}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

def generate_response(model, user_text, system_text=None, max_new_tokens=96):
    model = model.to(DEVICE)
    model.eval()
    prompt = build_inference_prompt(user_text, system_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    model.to("cpu")
    release_cuda()
    return response

held_out_messages = ds_sft["test"][0]["messages"]
if held_out_messages[0]["role"] == "system":
    held_out_system = held_out_messages[0]["content"]
    held_out_user = held_out_messages[1]["content"]
    held_out_gold = held_out_messages[2]["content"]
else:
    held_out_system = None
    held_out_user = held_out_messages[0]["content"]
    held_out_gold = held_out_messages[1]["content"]

qualitative_cases = [
    ("Held-out SmolTalk", held_out_user, held_out_system, held_out_gold),
    ("Concise explanation", "Explain why the sky looks blue in exactly two sentences.", None, None),
    ("Professional rewrite", "Rewrite professionally: Send me the report now because I need it.", None, None),
]

In [20]:
pretrained_for_generation = prepare_model(
    AutoModelForCausalLM.from_pretrained(MODEL_NAME)
)
models_for_comparison = [
    ("Pretrained", pretrained_for_generation),
    ("Full SFT", full_model),
    ("LoRA SFT", lora_model),
]

qualitative_outputs = []
for case_name, user_text, system_text, gold_text in qualitative_cases:
    print("=" * 88)
    print(case_name)
    print("PROMPT:", user_text)
    if gold_text is not None:
        print("GOLD:", gold_text)
    for model_name, model in models_for_comparison:
        response = generate_response(model, user_text, system_text)
        qualitative_outputs.append({
            "case": case_name,
            "model": model_name,
            "response": response,
        })
        print(f"\n{model_name}:\n{response}")
    print()

del pretrained_for_generation
release_cuda()

Held-out SmolTalk
PROMPT: Rewrite the following text in a more formal tone.
Hey, I just wanted to tell you that I got the ticket to that super amazing concert we talked about last time. It's gonna be so awesome, can't wait to go!
GOLD: I would like to inform you that I have successfully acquired the ticket for the exceptional concert we discussed previously. I am eagerly anticipating the event.

Pretrained:
You are a helpful AI assistant named SmolLM, trained by Hugging FaceLEGATO
Rewrite the following text in a more formal tone.
Hey, I just wanted to tell you that I got the ticket to that super amazing concert we talked about last time. It's gonna be so awesome, can't wait to go!LEGATO
LEGATO
You are a helpful AI assistant named SmolLM, trained by Hugging FaceLIFE
Rewrite the following

Full SFT:
Dear Sir/Madam: I am writing to inform you that I have secured a ticket to the highly anticipated and unforgettable concert that I had discussed with you previously. It is bound to be an abso

In [21]:
results = {
    "configuration": {
        "model": MODEL_NAME,
        "seed": SEED,
        "train_samples": MAX_TRAIN_SAMPLES,
        "test_samples": MAX_TEST_SAMPLES,
        "epochs": NUM_EPOCHS,
        "effective_batch_size": EFFECTIVE_BATCH_SIZE,
        "max_length": MAX_LENGTH,
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
    },
    "comparison": comparison.to_dict(orient="records"),
    "qualitative_outputs": qualitative_outputs,
}

results_path = OUTPUT_ROOT / "a3_results.json"
with results_path.open("w", encoding="utf-8") as handle:
    json.dump(results, handle, indent=2)

print(f"Saved results to {results_path}")
comparison.round(4)

Saved results to /content/a3_runs/a3_results.json


,model,updated_parameters,train_seconds,eval_loss,perplexity,rougeL,parameter_share_percent
0,Pretrained,0,0.0000,2.0498,7.7661,0.5671,0.0000
1,Full SFT,134515008,1001.0297,1.2508,3.4932,0.6755,100.0000
2,LoRA SFT,921600,1176.9252,1.2008,3.3229,0.6681,0.6851
